# Feature Engineering

#### Feature engineering is the process of selecting, transforming, creating, and refining input variables (features) so that a machine learning model can better understand patterns in the data.

 - Raw data → Cleaned & Transformed Data → Better Features → Better Model Performance
   
**Types of Feature Engineering:**

**Feature Transformation**
    
        Changing the format or scale of features:
           - Normalization / Standardization
           - Log transformation
           - Scaling
           - Encoding categorical variables
    
**Feature Encoding**
    
        Converting categorical data into numeric format:
        	- One-hot encoding
        	- Label encoding
        	- Target encoding
    
**Feature Creation**
    
        Creating new features from existing ones:
        	- Age groups from age
        	- Income per person
        	- Capital gain ratio
        	- Work experience buckets
    
**Feature Selection**
    
        Choosing only the most important features:
            - Correlation analysis
            - Mutual information
            - Feature importance (from tree models)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

In [2]:
df = pd.read_csv("df_train_test.csv")
train_df = df[df['source']=='train'].copy()
test_df = df[df['source']=='test'].copy()


In [3]:
df.sample(1)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,capital_gain_bin,capital_loss_bin,net_capital,income_binary
30046,31,Local-gov,158291,Bachelors,13,Never-married,Craft-repair,Not-in-family,White,Male,8614,0,40,United-States,>50K,train,1,0,8614,1


In [4]:
df["workclass"].value_counts()

workclass
Private             33879
Self-emp-not-inc     3861
Local-gov            3136
Unknown              2799
State-gov            1981
Self-emp-inc         1694
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64

In [5]:
df.loc[df["workclass"] == "Never-worked" , ["workclass","age","hours_per_week","income"]]

,workclass,age,hours_per_week,income
5359,Never-worked,18,40,<=50K
10842,Never-worked,23,35,<=50K
14767,Never-worked,17,30,<=50K
20328,Never-worked,18,10,<=50K
23217,Never-worked,20,40,<=50K
32281,Never-worked,30,40,<=50K
32291,Never-worked,18,4,<=50K
41321,Never-worked,17,20,<=50K
44141,Never-worked,20,35,<=50K
46431,Never-worked,18,35,<=50K


In [6]:
# age, edu_num - keep continous 


# Sex binary Transformation

In [7]:
# convert sex as binary MAle : 1 and Female : 0 
df["sex"] = df["sex"].map({'Male':1, 'Female': 0})


# Marital_status transformation . binary classification

In [8]:
df["marital_status"].value_counts()

marital_status
Married-civ-spouse       22372
Never-married            16098
Divorced                  6630
Separated                 1530
Widowed                   1518
Married-spouse-absent      628
Married-AF-spouse           37
Name: count, dtype: int64

In [9]:
# binary classification for marital_status

df['marital_status'] = df['marital_status'].map({
    'Married-civ-spouse': 1,
    'Married-spouse-absent': 1,
    'Married-AF-spouse': 1,
    'Never-married': 0,
    'Divorced': 0,
    'Separated': 0,
    'Widowed': 0
})

In [10]:
# Are they perfectly correlated? check each education string has exactly one numeric code
(df['education_num'].groupby(df['education']).nunique() == 1).all()

True

# Drop Education

In [11]:
# can safely drop education
df = df.drop(columns=['education'])

# Capital Gain and loss log transformation

In [12]:
# capital gain and loss has high right skewness where linear models requires transformation. retaining net_capital(raw with negative)
# Log transform skewed features
df['capital_gain'] = np.log1p(df['capital_gain'])
df['capital_loss'] = np.log1p(df['capital_loss'])


In [13]:
df['capital_gain'].head()

0    7.684784
1    0.000000
2    0.000000
3    0.000000
4    0.000000
Name: capital_gain, dtype: float64

# Drop Fnlwgt

In [14]:
df = df.drop(columns=['fnlwgt'])

# Hours per Week transformation 

In [15]:
df['hours_per_week'].describe()
df['hours_per_week'].value_counts().head(10)

hours_per_week
40    22787
50     4244
45     2716
60     2177
35     1935
20     1862
30     1699
55     1050
25      958
48      769
Name: count, dtype: int64

In [16]:
# normal labor distribution. Later, for Logistic Regression / SVM / KNN we can apply scaling based on results.

# frequenc encoding for workclass 

In [17]:
# 1️⃣ Compute frequency of each category using TRAIN data only
workclass_freq = train_df['workclass'].value_counts(normalize=True)

# 2️⃣ Map these frequencies to the combined dataframe
df['workclass'] = df['workclass'].map(workclass_freq)

# 3️⃣ Optional: fill any missing values (categories in test but not in train)
df['workclass'] = df['workclass'].fillna(0)

In [18]:
df['workclass'].head()

0    0.039893
1    0.078065
2    0.696837
3    0.696837
4    0.696837
Name: workclass, dtype: float64

In [19]:
# frequency encoding for occupation

In [20]:
# 1️⃣ Compute frequency on TRAIN data only
occupation_freq = train_df['occupation'].value_counts(normalize=True)

# 2️⃣ Map these frequencies to combined dataframe
df['occupation'] = df['occupation'].map(occupation_freq)

# 3️⃣ Handle any unseen categories in test (rare or new)
df['occupation'] = df['occupation'].fillna(0)

In [21]:
df['occupation'].sample(2)

7790     0.127117
10889    0.112180
Name: occupation, dtype: float64

In [22]:
df['race'].value_counts()

race
White                 41736
Black                  4683
Asian-Pac-Islander     1518
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64

### Race one hot encoding. merge the mer-indian-eskimo and other as these were < 1% 

In [23]:
# merge least 2 categories as other_race.
df["race"] = df["race"].replace({'Amer-Indian-Eskimo':'Other_Race','Other':'Other_Race'})

# one hot encode
race_dummies = pd.get_dummies(df["race"],prefix="race")

# drop one dummy col white as baseline
race_dummies = race_dummies.drop('race_White',axis=1)

#Attach back to dataframe
df = pd.concat([df,race_dummies], axis=1)

#drop original race col
df = df.drop('race', axis =1)

In [24]:
df.head()

,age,workclass,education_num,marital_status,occupation,relationship,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,capital_gain_bin,capital_loss_bin,net_capital,income_binary,race_Asian-Pac-Islander,race_Black,race_Other_Race
0,39,0.039893,13,0,0.115807,Not-in-family,1,7.684784,0.0,40,United-States,<=50K,train,1,0,2174,0,False,False,False
1,50,0.078065,13,1,0.124935,Husband,1,0.000000,0.0,13,United-States,<=50K,train,0,0,0,0,False,False,False
2,38,0.696837,9,0,0.042075,Not-in-family,1,0.000000,0.0,40,United-States,<=50K,train,0,0,0,0,False,False,False
3,53,0.696837,7,1,0.042075,Husband,1,0.000000,0.0,40,United-States,<=50K,train,0,0,0,0,False,True,False
4,28,0.696837,13,1,0.127117,Wife,0,0.000000,0.0,40,Cuba,<=50K,train,0,0,0,0,False,True,False


In [25]:
df["relationship"].value_counts()

relationship
Husband           19709
Not-in-family     12567
Own-child          7576
Unmarried          5124
Wife               2331
Other-relative     1506
Name: count, dtype: int64

# One hot encodign for relationship 

In [26]:
relationship_dummies = pd.get_dummies(df['relationship'], prefix='relationship')

# Drop baseline column (Not-in-family)
relationship_dummies = relationship_dummies.drop('relationship_Not-in-family', axis=1)

# Attach to df
df = pd.concat([df, relationship_dummies], axis=1)

# Drop original column
df = df.drop('relationship', axis=1)

In [27]:
df.head()

,age,workclass,education_num,marital_status,occupation,sex,capital_gain,capital_loss,hours_per_week,native_country,...,net_capital,income_binary,race_Asian-Pac-Islander,race_Black,race_Other_Race,relationship_Husband,relationship_Other-relative,relationship_Own-child,relationship_Unmarried,relationship_Wife
0,39,0.039893,13,0,0.115807,1,7.684784,0.0,40,United-States,...,2174,0,False,False,False,False,False,False,False,False
1,50,0.078065,13,1,0.124935,1,0.000000,0.0,13,United-States,...,0,0,False,False,False,True,False,False,False,False
2,38,0.696837,9,0,0.042075,1,0.000000,0.0,40,United-States,...,0,0,False,False,False,False,False,False,False,False
3,53,0.696837,7,1,0.042075,1,0.000000,0.0,40,United-States,...,0,0,False,True,False,True,False,False,False,False
4,28,0.696837,13,1,0.127117,0,0.000000,0.0,40,Cuba,...,0,0,False,True,False,False,False,False,False,True


# hours per week one hot encode after re-group as part time, over-time and full time

In [28]:
# Frequency encoding for native_country
country_freq = train_df['native_country'].value_counts(normalize=True)

df['native_country_freq'] = df['native_country'].map(country_freq)

df['native_country_freq'] = df['native_country_freq'].fillna(0)

In [29]:
# Create hours_per_week_category
def categorize_hours(x):
    if x < 20:
        return 'part-time'
    elif x <= 80:
        return 'full-time'
    else:
        return 'over-time'

df['hours_per_week_category'] = df['hours_per_week'].apply(categorize_hours)

In [30]:
# One-hot encode hours_per_week_category
hours_dummies = pd.get_dummies(df['hours_per_week_category'], prefix='hours')

# Drop baseline (full-time)
hours_dummies = hours_dummies.drop('hours_full-time', axis=1)

# Attach to df
df = pd.concat([df, hours_dummies], axis=1)

# Optional: drop original categorical column
df = df.drop('hours_per_week_category', axis=1)

In [37]:
df = df.drop('income' , axis=1)


KeyError: "['income'] not found in axis"

In [38]:
df.rename(columns={'income_binary': 'income'}, inplace=True)

In [39]:
# export cleaned features to csv 
# Export fully feature-engineered dataset to CSV
df.to_csv("df_train_test_cleaned.csv", index=False)

## Summary of Feature Engineering

**Kept as-is (numeric):**

   - age
    
   - education_num
    

**Mapped / encoded:**

   - gender → 0 = female, 1 = male
    
   - marital_status → 0 = unmarried, 1 = married
    
   - capital_gain / capital_loss → log transform, then net_capital = gain - loss
    
   - income → binary 0/1 + retained original column

**Frequency encoded:**

   - workclass
    
   - occupation
    
   - native_country (frequency calculated on train only and mapped to combined)

**One-hot encoded:**

   - race -- dropped race_White as baseline
    
   - relationship -- dropped Not-in-family as baseline

   - hours_per_week -- created a new categorical feature, one-hot encoded it, and dropped hours_full-time as the baseline.

**Dropped / removed:**

   - Original education (string)
    
  -  fnlwgt
    
  -  Original categorical columns after encoding (race, relationship, native_country)